# Teacher Interface

Four tools for a teacher:
1. **Add a New Task** -- create a task and attach sample/test input-output pairs.
2. **View & Download Tasks** -- browse existing tasks (filterable by theme/level/status) and export a task's description + sample data as a zip.
3. **View Submissions** -- browse student submissions, filterable by task, student, and status.
4. **Investigate a Submission** -- full detail on one submission: the student's code, the task, and every test result (including hidden ones).

Run all cells top to bottom once; each section's own controls can be reused afterwards.

In [ ]:
import sys
sys.path.insert(0, ".")

import ipywidgets as widgets
from IPython.display import display

import db

teachers = db.list_teachers()
topics = db.list_topics()
levels = db.list_levels()

print(f"{len(teachers)} teacher(s), {len(topics)} topic(s), {len(levels)} level(s) loaded.")

## Part 1: Add a New Task

### 1. Teacher
Pick yourself from the list, or register as a new teacher.

In [ ]:
def teacher_options():
    return [(f"{name} <{email}>", str(tid)) for tid, name, email in teachers]

teacher_dropdown = widgets.Dropdown(options=teacher_options(), description="Teacher:", style={"description_width": "100px"})
new_teacher_name = widgets.Text(description="Full name:", style={"description_width": "100px"})
new_teacher_email = widgets.Text(description="Email:", style={"description_width": "100px"})
add_teacher_btn = widgets.Button(description="+ Add teacher", button_style="")
teacher_out = widgets.Output()

def on_add_teacher(_):
    with teacher_out:
        teacher_out.clear_output()
        name, email = new_teacher_name.value.strip(), new_teacher_email.value.strip()
        if not name or not email:
            print("Full name and email are required.")
            return
        try:
            tid = db.create_teacher(name, email)
        except Exception as e:
            print(f"Could not add teacher: {e}")
            return
        teachers.append((tid, name, email))
        teacher_dropdown.options = teacher_options()
        teacher_dropdown.value = str(tid)
        new_teacher_name.value = ""
        new_teacher_email.value = ""
        print(f"Added teacher {name}.")

add_teacher_btn.on_click(on_add_teacher)

display(widgets.VBox([
    teacher_dropdown,
    widgets.HBox([new_teacher_name, new_teacher_email, add_teacher_btn]),
    teacher_out,
]))

### 2. Topic
Pick an existing topic, or create a new one.

In [ ]:
def topic_options():
    return [(name, str(tid)) for tid, name, slug, parent_id in topics]

topic_dropdown = widgets.Dropdown(options=topic_options(), description="Topic:", style={"description_width": "100px"})
new_topic_name = widgets.Text(description="New topic:", style={"description_width": "100px"})
add_topic_btn = widgets.Button(description="+ Add topic")
topic_out = widgets.Output()

def on_add_topic(_):
    with topic_out:
        topic_out.clear_output()
        name = new_topic_name.value.strip()
        if not name:
            print("Topic name is required.")
            return
        try:
            tid = db.create_topic(name)
        except Exception as e:
            print(f"Could not add topic: {e}")
            return
        topics.append((tid, name, None, None))
        topic_dropdown.options = topic_options()
        topic_dropdown.value = str(tid)
        new_topic_name.value = ""
        print(f"Added topic {name}.")

add_topic_btn.on_click(on_add_topic)

display(widgets.VBox([
    topic_dropdown,
    widgets.HBox([new_topic_name, add_topic_btn]),
    topic_out,
]))


### 3. Task details

In [ ]:
level_dropdown = widgets.Dropdown(
    options=[(name, str(lid)) for lid, code, name in levels],
    description="Level:",
    style={"description_width": "100px"},
)
title_text = widgets.Text(description="Title:", style={"description_width": "100px"}, layout=widgets.Layout(width="600px"))
description_text = widgets.Textarea(
    description="Description:",
    placeholder="Markdown supported",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px", height="150px"),
)
status_dropdown = widgets.Dropdown(
    options=["draft", "published", "archived"],
    value="draft",
    description="Status:",
    style={"description_width": "100px"},
)

display(widgets.VBox([level_dropdown, title_text, description_text, status_dropdown]))


### 4. Files
Each test case needs a matching **input** file and **output** (expected result) file. Upload the input files and the output files for a given type together — they're paired up in filename order, so e.g. `case1_in.txt`/`case1_out.txt` and `case2_in.txt`/`case2_out.txt` should sort the same way in both boxes. **Sample** cases are shown to students; **test** cases are hidden and used for auto-grading.

In [ ]:
sample_input_upload = widgets.FileUpload(description="Sample input(s)", multiple=True)
sample_output_upload = widgets.FileUpload(description="Sample output(s)", multiple=True)
test_input_upload = widgets.FileUpload(description="Test input(s)", multiple=True)
test_output_upload = widgets.FileUpload(description="Test output(s)", multiple=True)

display(widgets.VBox([
    widgets.HBox([widgets.Label("Sample input:", layout=widgets.Layout(width="120px")), sample_input_upload]),
    widgets.HBox([widgets.Label("Sample output:", layout=widgets.Layout(width="120px")), sample_output_upload]),
    widgets.HBox([widgets.Label("Test input:", layout=widgets.Layout(width="120px")), test_input_upload]),
    widgets.HBox([widgets.Label("Test output:", layout=widgets.Layout(width="120px")), test_output_upload]),
]))

### 5. Submit

In [ ]:
submit_btn = widgets.Button(description="Create task", button_style="success")
submit_out = widgets.Output()

def uploaded_items(upload_widget):
    # ipywidgets>=8 gives .value as a tuple of dicts; older gives a dict keyed by filename.
    value = upload_widget.value
    if isinstance(value, dict):
        items = [(name, item["content"].tobytes() if hasattr(item["content"], "tobytes") else bytes(item["content"])) for name, item in value.items()]
    else:
        items = [(item["name"], item["content"].tobytes() if hasattr(item["content"], "tobytes") else bytes(item["content"])) for item in value]
    return sorted(items, key=lambda pair: pair[0])  # sort by filename so input/output boxes pair up positionally

def on_submit(_):
    with submit_out:
        submit_out.clear_output()

        title = title_text.value.strip()
        description = description_text.value.strip()
        if not title or not description:
            print("Title and description are required.")
            return
        if not teacher_dropdown.options or not topic_dropdown.options:
            print("Need at least one teacher and one topic before creating a task.")
            return

        cases = {}
        for dataset_type, in_widget, out_widget in (
            ("sample", sample_input_upload, sample_output_upload),
            ("test", test_input_upload, test_output_upload),
        ):
            inputs = uploaded_items(in_widget)
            outputs = uploaded_items(out_widget)
            if len(inputs) != len(outputs):
                print(f"{dataset_type}: {len(inputs)} input file(s) but {len(outputs)} output file(s) -- counts must match.")
                return
            cases[dataset_type] = list(zip(inputs, outputs))

        author_id = teacher_dropdown.value
        topic_id = topic_dropdown.value
        level_id = level_dropdown.value
        status = status_dropdown.value

        try:
            task_id = db.create_task(title, description, topic_id, level_id, author_id, status)
        except Exception as e:
            print(f"Failed to create task: {e}")
            return

        print(f"Created task {task_id} ({title!r}, status={status}).")

        for dataset_type, pairs in cases.items():
            for order_index, ((in_name, in_content), (out_name, out_content)) in enumerate(pairs):
                try:
                    in_meta = db.upload_file(in_content, in_name, task_id)
                    input_file_id = db.register_file(in_meta, uploaded_by=author_id)
                    out_meta = db.upload_file(out_content, out_name, task_id)
                    output_file_id = db.register_file(out_meta, uploaded_by=author_id)
                    db.attach_dataset(task_id, input_file_id, output_file_id, dataset_type, order_index)
                    print(f"  attached {dataset_type} case {order_index}: {in_name} -> {out_name}")
                except Exception as e:
                    print(f"  failed to attach {dataset_type} case {order_index} ({in_name} -> {out_name}): {e}")

        print("Done.")

submit_btn.on_click(on_submit)
display(widgets.VBox([submit_btn, submit_out]))

## Part 2: View & Download Tasks

In [ ]:
import html as _html
import uuid as _uuid
from IPython.display import HTML

def render_table(rows, columns, sortable=True):
    """Renders rows as an HTML table. If sortable, clicking a header sorts by
    that column (numeric if every cell in it parses as a number, else text),
    toggling ascending/descending on repeat clicks -- plain vanilla JS, no
    library, scoped to this table's own id so multiple tables on the page
    don't interfere with each other."""
    if not rows:
        return HTML("<i>No rows.</i>")

    table_id = f"tbl_{_uuid.uuid4().hex}"

    def header_cell(i, c):
        label = _html.escape(str(c))
        if not sortable:
            return f"<th style='text-align:left; padding:4px 10px'>{label}</th>"
        return (
            f"<th onclick=\"_sortTable('{table_id}', {i})\" "
            f"style='text-align:left; padding:4px 10px; cursor:pointer; user-select:none' "
            f"title='Click to sort'>{label} ⇅</th>"
        )

    head = "".join(header_cell(i, c) for i, c in enumerate(columns))
    body = "".join(
        "<tr>" + "".join(
            f"<td style='padding:4px 10px'>{_html.escape(str(row.get(c, '')))}</td>" for c in columns
        ) + "</tr>"
        for row in rows
    )
    script = "" if not sortable else """
<script>
window._sortTable = window._sortTable || function(tableId, colIdx) {
    var table = document.getElementById(tableId);
    var tbody = table.tBodies[0];
    var rows = Array.prototype.slice.call(tbody.rows);
    var asc = !(table.getAttribute('data-sort-col') == colIdx && table.getAttribute('data-sort-dir') === 'asc');
    function cellText(row) { return row.cells[colIdx].innerText.trim(); }
    var allNumeric = rows.every(function(r) { var t = cellText(r); return t === '' || !isNaN(parseFloat(t)); });
    rows.sort(function(a, b) {
        var ta = cellText(a), tb = cellText(b), cmp;
        cmp = allNumeric ? ((parseFloat(ta) || 0) - (parseFloat(tb) || 0)) : ta.localeCompare(tb);
        return asc ? cmp : -cmp;
    });
    rows.forEach(function(r) { tbody.appendChild(r); });
    table.setAttribute('data-sort-col', colIdx);
    table.setAttribute('data-sort-dir', asc ? 'asc' : 'desc');
};
</script>
"""
    return HTML(f"<table id='{table_id}' style='border-collapse:collapse'><tr>{head}</tr>{body}</table>{script}")


def task_theme_filter_options():
    return [("All themes", None)] + [(t["name"], t["id"]) for t in topics]

def task_level_filter_options():
    return [("All levels", None)] + [(l["name"], l["id"]) for l in levels]

task_status_options = ["All", "draft", "published", "archived"]

task_theme_filter = widgets.Dropdown(options=task_theme_filter_options(), description="Theme:", style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
task_level_filter = widgets.Dropdown(options=task_level_filter_options(), description="Level:", style={"description_width": "100px"}, layout=widgets.Layout(width="250px"))
task_status_filter = widgets.Dropdown(options=task_status_options, description="Status:", style={"description_width": "100px"}, layout=widgets.Layout(width="250px"))

tasks_out = widgets.Output()
refresh_tasks_btn = widgets.Button(description="Refresh tasks")

def on_refresh_tasks(_):
    with tasks_out:
        tasks_out.clear_output()
        status = None if task_status_filter.value == "All" else task_status_filter.value
        rows = db.list_tasks(topic_id=task_theme_filter.value, level_id=task_level_filter.value, status=status)
        display(render_table(rows, ["id", "title", "topic", "level", "status", "created_at"]))

refresh_tasks_btn.on_click(on_refresh_tasks)
on_refresh_tasks(None)

display(widgets.VBox([
    widgets.HBox([task_theme_filter, task_level_filter, task_status_filter]),
    refresh_tasks_btn,
    tasks_out,
]))

### Download a task
Paste a task id from the table above, then prepare a zip with the task description plus its **sample** input/output files. Hidden test files are never included, even here.

In [ ]:
import base64

download_task_id = widgets.Text(description="Task id:", style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
download_btn = widgets.Button(description="Prepare download", button_style="info")
download_status = widgets.HTML(value="")

def on_download(_):
    task_id = download_task_id.value.strip()
    if not task_id:
        download_status.value = "<span style='color:#b00020'>Enter a task id.</span>"
        return
    try:
        zip_path = db.build_task_package(task_id)
    except Exception as e:
        download_status.value = f"<span style='color:#b00020'>Could not build package: {e}</span>"
        return

    data = zip_path.read_bytes()
    warning = ""
    if len(data) > 10_000_000:
        warning = (
            f"<div style='color:#b00020'>Warning: {len(data)/1e6:.1f} MB is large "
            f"for an in-browser download link; it may be slow.</div>"
        )

    # Embed the file as a data: URI so the browser downloads it straight
    # from this link's own content -- no request back to wherever this
    # notebook's kernel/server is running, so it lands on whatever
    # machine the browser (and this click) are on, not the kernel's.
    b64 = base64.b64encode(data).decode("ascii")
    href = f"data:application/zip;base64,{b64}"

    # A plain HTML widget's .value is set once and just sits there -- no
    # Output-widget clear/append cycle that could race and blank itself.
    download_status.value = (
        warning
        + f'<a download="{zip_path.name}" href="{href}" '
        f'style="display:inline-block;padding:6px 14px;background:#2e7d32;color:white;'
        f'border-radius:4px;text-decoration:none;font-family:sans-serif">'
        f'⬇ Download {zip_path.name} ({len(data)/1024:.1f} KB)</a>'
        f'<div style="margin-top:6px;color:#666;font-size:0.9em">'
        f'Also saved on the machine running this notebook at: {zip_path}</div>'
    )

download_btn.on_click(on_download)
display(widgets.VBox([download_task_id, download_btn, download_status]))

## Part 3: View Submissions
Filter by task, student, and/or status -- leave a filter on "All" to ignore it.

In [ ]:
status_options = ["All", "submitted", "checking", "checked", "needs_review", "rejected"]

def task_filter_options():
    return [("All tasks", None)] + [(f"{t['title']} ({t['id']})", t["id"]) for t in db.list_tasks()]

def student_filter_options():
    return [("All students", None)] + [(f"{s['full_name']} <{s['email']}>", s["id"]) for s in db.list_students()]

task_filter = widgets.Dropdown(options=task_filter_options(), description="Task:", style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
student_filter = widgets.Dropdown(options=student_filter_options(), description="Student:", style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
status_filter = widgets.Dropdown(options=status_options, description="Status:", style={"description_width": "100px"})
refresh_submissions_btn = widgets.Button(description="Refresh submissions")
submissions_out = widgets.Output()

def on_refresh_submissions(_):
    with submissions_out:
        submissions_out.clear_output()
        status = None if status_filter.value == "All" else status_filter.value
        rows = db.list_submissions(task_id=task_filter.value, student_id=student_filter.value, status=status)
        for r in rows:
            r["score"] = "" if r["auto_score"] is None else f"{r['auto_score']}/{r['auto_max_score']}"
        display(render_table(rows, ["submitted_at", "student_name", "student_email", "task_title", "status", "score"]))

refresh_submissions_btn.on_click(on_refresh_submissions)
on_refresh_submissions(None)

display(widgets.VBox([
    widgets.HBox([task_filter, student_filter, status_filter]),
    refresh_submissions_btn,
    submissions_out,
]))

## Part 4: Investigate a Submission
Everything for one submission -- status, LLM feedback, the student's code, the task it was for, and every test result (including hidden `test` cases, with full stdout/stderr). Paste a submission id from the table above.

In [ ]:
from IPython.display import Markdown

inspect_submission_id = widgets.Text(description="Submission id:", style={"description_width": "120px"}, layout=widgets.Layout(width="500px"))
inspect_btn = widgets.Button(description="Inspect", button_style="info")
inspect_out = widgets.Output()

def on_inspect(_):
    with inspect_out:
        inspect_out.clear_output()
        sub_id = inspect_submission_id.value.strip()
        if not sub_id:
            print("Enter a submission id.")
            return

        detail = db.get_submission_detail(sub_id)
        if detail is None:
            print("No submission with that id.")
            return

        s, task, student, code, results = (
            detail["submission"], detail["task"], detail["student"], detail["code"], detail["test_results"]
        )

        display(Markdown(
            f"### Submission `{s['id']}`\n"
            f"- **Status:** {s['status']}\n"
            f"- **Student:** {student['full_name']} <{student['email']}>\n"
            f"- **Task:** {task['title']} (`{task['id']}`)\n"
            f"- **Submitted:** {s['submitted_at']}\n"
            f"- **Auto score:** {s['auto_score']} / {s['auto_max_score']}"
            f" (model: {s['auto_feedback_model'] or '-'})\n"
            f"- **Human score:** {s['human_score'] if s['human_score'] is not None else '-'}\n"
        ))

        feedback = s["auto_feedback"] or "_none yet_"
        display(Markdown("**LLM feedback:**\n\n> " + feedback.replace("\n", "\n> ")))

        display(Markdown(f"**Task description:**\n\n{task['description']}"))

        display(Markdown("**Student code:**"))
        display(Markdown(f"```python\n{code or '(no code file)'}\n```"))

        display(Markdown("**Test results (including hidden test cases):**"))
        display(render_table(
            [{**r, "stdout": (r["stdout"] or "")[:1000], "stderr": (r["stderr"] or "")[:1000]} for r in results],
            ["dataset_type", "passed", "exit_code", "execution_time_ms", "stdout", "stderr"],
            sortable=False,
        ))

inspect_btn.on_click(on_inspect)
display(widgets.VBox([inspect_submission_id, inspect_btn, inspect_out]))